# StackLite Hybrid QA Demo

        This Colab-ready notebook builds a technical question-answering assistant over the provided StackLite JSON corpus.

        It demonstrates BM25 retrieval, MiniLM dense retrieval, RRF hybrid fusion, retrieval metrics, RAG answer generation with citations, and a Gradio UI.

## 1. Setup

        In Google Colab, upload `DataSet.zip` to the session or mount Drive and update `DATA_ZIP`.
        Use a free GPU runtime for faster MiniLM embeddings and HuggingFace generation.

In [ ]:
!pip -q install beautifulsoup4 gradio numpy openai pandas rank-bm25 scikit-learn sentence-transformers torch transformers tqdm

In [ ]:
from pathlib import Path
        import shutil
        import zipfile

        PROJECT_DIR = Path.cwd() / "stacklite_hybrid_qa"
        DATA_ZIP = Path("/content/DataSet.zip")

        PROJECT_DIR.mkdir(exist_ok=True)
        (PROJECT_DIR / "data").mkdir(exist_ok=True)

        if DATA_ZIP.exists():
            with zipfile.ZipFile(DATA_ZIP) as zf:
                zf.extractall(PROJECT_DIR / "data")
        else:
            print("Upload DataSet.zip to /content/DataSet.zip, or run this notebook from the submitted repo where data/ already exists.")

## 2. Core Code

This cell writes the same project module used by the repository so the notebook can run standalone in Colab.

In [ ]:
import sys
        from pathlib import Path

        Path("stacklite_qa").mkdir(exist_ok=True)
        Path("evaluation").mkdir(exist_ok=True)
        Path("stacklite_qa/core.py").write_text('from __future__ import annotations\n\nimport json\nimport math\nimport os\nimport re\nfrom dataclasses import dataclass\nfrom html import unescape\nfrom pathlib import Path\nfrom typing import Callable, Iterable\n\nimport numpy as np\n\ntry:\n    from bs4 import BeautifulSoup\nexcept ImportError:  # pragma: no cover - used only in minimal environments.\n    BeautifulSoup = None\n\n\nTOKEN_RE = re.compile(r"[A-Za-z0-9_+#.\\-]+")\n\n\n@dataclass(frozen=True)\nclass Document:\n    doc_id: str\n    question_id: int\n    source: str\n    title: str\n    body: str\n    tags: list[str]\n    link: str\n    score: int\n\n    @property\n    def text(self) -> str:\n        tags = " ".join(self.tags)\n        return f"{self.title}\\nTags: {tags}\\n{self.body}"\n\n    @property\n    def citation(self) -> str:\n        return f"[{self.doc_id}] {self.title} ({self.link})"\n\n\n@dataclass\nclass SearchResult:\n    doc: Document\n    score: float\n    rank: int\n    bm25_score: float | None = None\n    dense_score: float | None = None\n\n\ndef clean_html(raw_html: str) -> str:\n    if BeautifulSoup is not None:\n        text = BeautifulSoup(raw_html or "", "html.parser").get_text(" ")\n    else:\n        text = re.sub(r"<[^>]+>", " ", raw_html or "")\n    return re.sub(r"\\s+", " ", unescape(text)).strip()\n\n\ndef tokenize(text: str) -> list[str]:\n    return [tok.lower() for tok in TOKEN_RE.findall(text or "")]\n\n\ndef load_stacklite_documents(data_dir: str | Path = "data") -> list[Document]:\n    data_path = Path(data_dir)\n    files = sorted(data_path.glob("top_*_questions.json"))\n    if not files:\n        raise FileNotFoundError(f"No StackLite JSON files found in {data_path.resolve()}")\n\n    docs: list[Document] = []\n    for file_path in files:\n        source = "ai" if "ai" in file_path.stem else "datascience"\n        with file_path.open(encoding="utf-8") as f:\n            records = json.load(f)\n\n        for row in records:\n            qid = int(row["question_id"])\n            title = unescape(str(row.get("title", ""))).strip()\n            tags = [str(tag) for tag in row.get("tags", [])]\n            docs.append(\n                Document(\n                    doc_id=f"{source}:{qid}",\n                    question_id=qid,\n                    source=source,\n                    title=title,\n                    body=clean_html(str(row.get("body", ""))),\n                    tags=tags,\n                    link=str(row.get("link", "")),\n                    score=int(row.get("score", 0) or 0),\n                )\n            )\n    return docs\n\n\nclass StackLiteHybridQA:\n    def __init__(\n        self,\n        documents: Iterable[Document],\n        embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",\n    ) -> None:\n        self.documents = list(documents)\n        if not self.documents:\n            raise ValueError("At least one document is required.")\n        self.embedding_model_name = embedding_model_name\n        self._doc_by_id = {doc.doc_id: doc for doc in self.documents}\n        self._bm25 = None\n        self._tokenized_corpus: list[list[str]] | None = None\n        self._embedding_model = None\n        self._doc_embeddings: np.ndarray | None = None\n\n    def build_bm25(self) -> None:\n        try:\n            from rank_bm25 import BM25Okapi\n        except ImportError as exc:\n            raise ImportError("Install rank-bm25 to use BM25 retrieval.") from exc\n        self._tokenized_corpus = [tokenize(doc.text) for doc in self.documents]\n        self._bm25 = BM25Okapi(self._tokenized_corpus)\n\n    def build_dense(self, batch_size: int = 64) -> None:\n        try:\n            from sentence_transformers import SentenceTransformer\n        except ImportError as exc:\n            raise ImportError("Install sentence-transformers to use dense retrieval.") from exc\n        self._embedding_model = SentenceTransformer(self.embedding_model_name)\n        embeddings = self._embedding_model.encode(\n            [doc.text for doc in self.documents],\n            batch_size=batch_size,\n            convert_to_numpy=True,\n            normalize_embeddings=True,\n            show_progress_bar=True,\n        )\n        self._doc_embeddings = np.asarray(embeddings, dtype=np.float32)\n\n    def bm25_search(self, query: str, top_k: int = 10) -> list[SearchResult]:\n        if self._bm25 is None:\n            self.build_bm25()\n        assert self._bm25 is not None\n        scores = np.asarray(self._bm25.get_scores(tokenize(query)), dtype=np.float32)\n        order = np.argsort(scores)[::-1][:top_k]\n        return [\n            SearchResult(doc=self.documents[i], score=float(scores[i]), rank=rank, bm25_score=float(scores[i]))\n            for rank, i in enumerate(order, start=1)\n        ]\n\n    def dense_search(self, query: str, top_k: int = 10) -> list[SearchResult]:\n        if self._doc_embeddings is None or self._embedding_model is None:\n            self.build_dense()\n        assert self._embedding_model is not None and self._doc_embeddings is not None\n        query_vec = self._embedding_model.encode(\n            [query],\n            convert_to_numpy=True,\n            normalize_embeddings=True,\n            show_progress_bar=False,\n        )[0]\n        scores = np.dot(self._doc_embeddings, query_vec.astype(np.float32))\n        order = np.argsort(scores)[::-1][:top_k]\n        return [\n            SearchResult(doc=self.documents[i], score=float(scores[i]), rank=rank, dense_score=float(scores[i]))\n            for rank, i in enumerate(order, start=1)\n        ]\n\n    def rrf_fuse(\n        self,\n        ranked_lists: list[list[SearchResult]],\n        top_k: int = 10,\n        rrf_k: int = 60,\n    ) -> list[SearchResult]:\n        combined: dict[str, SearchResult] = {}\n        scores: dict[str, float] = {}\n        for results in ranked_lists:\n            for result in results:\n                doc_id = result.doc.doc_id\n                scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + result.rank)\n                if doc_id not in combined:\n                    combined[doc_id] = SearchResult(doc=result.doc, score=0.0, rank=0)\n                if result.bm25_score is not None:\n                    combined[doc_id].bm25_score = result.bm25_score\n                if result.dense_score is not None:\n                    combined[doc_id].dense_score = result.dense_score\n\n        ordered = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]\n        fused: list[SearchResult] = []\n        for rank, (doc_id, score) in enumerate(ordered, start=1):\n            result = combined[doc_id]\n            result.score = float(score)\n            result.rank = rank\n            fused.append(result)\n        return fused\n\n    def hybrid_search(self, query: str, top_k: int = 10, candidate_k: int = 50) -> list[SearchResult]:\n        bm25 = self.bm25_search(query, top_k=candidate_k)\n        dense = self.dense_search(query, top_k=candidate_k)\n        return self.rrf_fuse([bm25, dense], top_k=top_k)\n\n    def format_contexts(self, results: list[SearchResult], max_chars_per_doc: int = 900) -> str:\n        chunks = []\n        for result in results:\n            doc = result.doc\n            text = doc.body[:max_chars_per_doc].strip()\n            chunks.append(\n                f"Source {result.rank}: {doc.doc_id}\\n"\n                f"Title: {doc.title}\\n"\n                f"Tags: {\', \'.join(doc.tags)}\\n"\n                f"URL: {doc.link}\\n"\n                f"Passage: {text}"\n            )\n        return "\\n\\n".join(chunks)\n\n    def generate_answer(\n        self,\n        query: str,\n        results: list[SearchResult],\n        backend: str = "extractive",\n        model_name: str = "google/flan-t5-base",\n        max_new_tokens: int = 220,\n    ) -> str:\n        if backend == "openai":\n            return self._generate_openai(query, results, max_new_tokens=max_new_tokens)\n        if backend == "hf":\n            return self._generate_hf(query, results, model_name=model_name, max_new_tokens=max_new_tokens)\n        return self._generate_extractive(query, results)\n\n    def _generate_hf(\n        self,\n        query: str,\n        results: list[SearchResult],\n        model_name: str,\n        max_new_tokens: int,\n    ) -> str:\n        try:\n            from transformers import pipeline\n        except ImportError as exc:\n            raise ImportError("Install transformers and torch to use the HuggingFace RAG backend.") from exc\n        context = self.format_contexts(results[:4], max_chars_per_doc=700)\n        prompt = (\n            "Answer the technical question using only the sources below. "\n            "Cite source ids like [datascience:15989] after each factual claim.\\n\\n"\n            f"Question: {query}\\n\\nSources:\\n{context}\\n\\nAnswer:"\n        )\n        generator = pipeline("text2text-generation", model=model_name)\n        answer = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"].strip()\n        citations = "\\n".join(f"- {result.doc.citation}" for result in results[:4])\n        return f"{answer}\\n\\nCitations:\\n{citations}"\n\n    def _generate_openai(self, query: str, results: list[SearchResult], max_new_tokens: int) -> str:\n        try:\n            from openai import OpenAI\n        except ImportError as exc:\n            raise ImportError("Install openai and set OPENAI_API_KEY to use the OpenAI backend.") from exc\n        if not os.getenv("OPENAI_API_KEY"):\n            raise RuntimeError("OPENAI_API_KEY is not set.")\n        client = OpenAI()\n        context = self.format_contexts(results[:5], max_chars_per_doc=900)\n        response = client.chat.completions.create(\n            model=os.getenv("OPENAI_MODEL", "gpt-3.5-turbo"),\n            messages=[\n                {\n                    "role": "system",\n                    "content": (\n                        "You are a careful technical QA assistant. Use only the provided StackLite sources. "\n                        "Cite source ids in square brackets for claims."\n                    ),\n                },\n                {"role": "user", "content": f"Question: {query}\\n\\nSources:\\n{context}"},\n            ],\n            max_tokens=max_new_tokens,\n            temperature=0,\n        )\n        answer = response.choices[0].message.content.strip()\n        citations = "\\n".join(f"- {result.doc.citation}" for result in results[:5])\n        return f"{answer}\\n\\nCitations:\\n{citations}"\n\n    def _generate_extractive(self, query: str, results: list[SearchResult]) -> str:\n        query_terms = set(tokenize(query))\n        selected = []\n        for result in results[:3]:\n            sentences = re.split(r"(?<=[.!?])\\s+", result.doc.body)\n            best = max(\n                sentences[:16] or [result.doc.body],\n                key=lambda sent: len(query_terms.intersection(tokenize(sent))) / math.sqrt(len(tokenize(sent)) + 1),\n            )\n            selected.append(f"{best.strip()} [{result.doc.doc_id}]")\n        answer = " ".join(part for part in selected if part)\n        citations = "\\n".join(f"- {result.doc.citation}" for result in results[:3])\n        return f"{answer}\\n\\nCitations:\\n{citations}"\n\n\ndef load_evaluation_questions(path: str | Path = "evaluation/eval_questions.json") -> list[dict]:\n    with Path(path).open(encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef _average_precision_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:\n    if not relevant:\n        return 0.0\n    hits = 0\n    total = 0.0\n    for idx, doc_id in enumerate(retrieved[:k], start=1):\n        if doc_id in relevant:\n            hits += 1\n            total += hits / idx\n    return total / min(len(relevant), k)\n\n\ndef _reciprocal_rank_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:\n    for idx, doc_id in enumerate(retrieved[:k], start=1):\n        if doc_id in relevant:\n            return 1.0 / idx\n    return 0.0\n\n\ndef _dcg_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:\n    return sum((1.0 / math.log2(idx + 1)) for idx, doc_id in enumerate(retrieved[:k], start=1) if doc_id in relevant)\n\n\ndef _ndcg_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:\n    ideal_hits = min(len(relevant), k)\n    if ideal_hits == 0:\n        return 0.0\n    ideal = sum(1.0 / math.log2(idx + 1) for idx in range(1, ideal_hits + 1))\n    return _dcg_at_k(retrieved, relevant, k) / ideal\n\n\ndef evaluate_retriever(\n    questions: list[dict],\n    search_fn: Callable[[str, int], list[SearchResult]],\n    k: int = 10,\n) -> dict[str, float]:\n    ap_scores = []\n    rr_scores = []\n    ndcg_scores = []\n    hit_scores = []\n    for item in questions:\n        relevant = set(item["relevant_doc_ids"])\n        results = search_fn(item["question"], k)\n        retrieved = [result.doc.doc_id for result in results]\n        ap_scores.append(_average_precision_at_k(retrieved, relevant, k))\n        rr_scores.append(_reciprocal_rank_at_k(retrieved, relevant, k))\n        ndcg_scores.append(_ndcg_at_k(retrieved, relevant, k))\n        hit_scores.append(float(bool(relevant.intersection(retrieved[:k]))))\n    return {\n        f"MAP@{k}": float(np.mean(ap_scores)),\n        f"MRR@{k}": float(np.mean(rr_scores)),\n        f"nDCG@{k}": float(np.mean(ndcg_scores)),\n        f"HitRate@{k}": float(np.mean(hit_scores)),\n    }\n', encoding="utf-8")
        Path("stacklite_qa/__init__.py").write_text('"""StackLite hybrid retrieval and RAG helpers."""\n\nfrom .core import (\n    Document,\n    SearchResult,\n    StackLiteHybridQA,\n    evaluate_retriever,\n    load_evaluation_questions,\n    load_stacklite_documents,\n)\n\n__all__ = [\n    "Document",\n    "SearchResult",\n    "StackLiteHybridQA",\n    "evaluate_retriever",\n    "load_evaluation_questions",\n    "load_stacklite_documents",\n]\n', encoding="utf-8")
        Path("evaluation/eval_questions.json").write_text('[\n  {\n    "id": "eval_01",\n    "question": "How should I interpret micro average versus macro average metrics for imbalanced multiclass classification?",\n    "relevant_doc_ids": ["datascience:15989"],\n    "expected_citation": "Micro Average vs Macro average Performance in a Multiclass classification setting"\n  },\n  {\n    "id": "eval_02",\n    "question": "What is the difference between fit and fit_transform in scikit-learn preprocessing or modeling?",\n    "relevant_doc_ids": ["datascience:12321"],\n    "expected_citation": "What\'s the difference between fit and fit_transform in scikit-learn models?"\n  },\n  {\n    "id": "eval_03",\n    "question": "How can Keras class weights help when the training data has imbalanced classes?",\n    "relevant_doc_ids": ["datascience:13490"],\n    "expected_citation": "How to set class weights for imbalanced classes in Keras?"\n  },\n  {\n    "id": "eval_04",\n    "question": "When should I use one hot encoding instead of label encoding for categorical variables?",\n    "relevant_doc_ids": ["datascience:9443"],\n    "expected_citation": "When to use One Hot Encoding vs LabelEncoder vs DictVectorizor?"\n  },\n  {\n    "id": "eval_05",\n    "question": "Why do transformer models need positional encodings?",\n    "relevant_doc_ids": ["datascience:51065"],\n    "expected_citation": "What is the positional encoding in the transformer model?"\n  },\n  {\n    "id": "eval_06",\n    "question": "What is the practical difference between model-based and model-free reinforcement learning?",\n    "relevant_doc_ids": ["ai:4456"],\n    "expected_citation": "What\'s the difference between model-free and model-based reinforcement learning?"\n  },\n  {\n    "id": "eval_07",\n    "question": "What does self-supervised learning mean in machine learning?",\n    "relevant_doc_ids": ["ai:10623"],\n    "expected_citation": "What is self-supervised learning in machine learning?"\n  },\n  {\n    "id": "eval_08",\n    "question": "Why can transformers handle long-range dependencies better than RNNs or LSTMs?",\n    "relevant_doc_ids": ["ai:20075"],\n    "expected_citation": "Why does the transformer do better than RNN and LSTM in long-range context dependencies?"\n  },\n  {\n    "id": "eval_09",\n    "question": "How does ChatGPT keep track of context from previous questions in a conversation?",\n    "relevant_doc_ids": ["ai:38150"],\n    "expected_citation": "How does ChatGPT retain the context of previous questions?"\n  },\n  {\n    "id": "eval_10",\n    "question": "What does temperature control in GPT style language models?",\n    "relevant_doc_ids": ["ai:32477"],\n    "expected_citation": "What is the \\"temperature\\" in the GPT models?"\n  }\n]\n', encoding="utf-8")
        sys.path.insert(0, str(Path.cwd()))

## 3. Load Corpus

In [ ]:
import sys
        sys.path.insert(0, str(Path.cwd()))

        from stacklite_qa import StackLiteHybridQA, evaluate_retriever, load_evaluation_questions, load_stacklite_documents

        data_dir = Path("data") if Path("data/top_ai_questions.json").exists() else PROJECT_DIR / "data"
        docs = load_stacklite_documents(data_dir)
        engine = StackLiteHybridQA(docs)
        print(f"Loaded {len(docs)} StackLite documents")
        print(docs[0])

## 4. BM25 Retrieval: Top-10 Results

In [ ]:
sample_question = "What is the difference between fit and fit_transform in scikit-learn?"
        bm25_results = engine.bm25_search(sample_question, top_k=10)
        for result in bm25_results:
            print(f"{result.rank:02d} | {result.doc.doc_id} | {result.score:.2f} | {result.doc.title}")
            print(f"     {result.doc.link}")

## 5. Retrieval Evaluation: MAP@10 and MRR@10

In [ ]:
eval_file = Path("evaluation/eval_questions.json")
        questions = load_evaluation_questions(eval_file)
        bm25_metrics = evaluate_retriever(questions, lambda q, k: engine.bm25_search(q, top_k=k), k=10)
        bm25_metrics

## 6. Dense Retrieval with MiniLM and nDCG@10

In [ ]:
engine.build_dense()
        dense_metrics = evaluate_retriever(questions, lambda q, k: engine.dense_search(q, top_k=k), k=10)
        dense_metrics

## 7. Hybrid RRF Retrieval

In [ ]:
hybrid_metrics = evaluate_retriever(questions, lambda q, k: engine.hybrid_search(q, top_k=k), k=10)
        print("BM25:", bm25_metrics)
        print("Dense MiniLM:", dense_metrics)
        print("Hybrid RRF:", hybrid_metrics)

## 8. RAG Answer with Citations

In [ ]:
rag_question = "What is the positional encoding in the transformer model?"
        contexts = engine.hybrid_search(rag_question, top_k=5)

        # Use backend='hf' for an open-source LLM answer. The extractive fallback is faster for class demos.
        answer = engine.generate_answer(rag_question, contexts, backend="extractive")
        print(answer)

## 9. Optional HuggingFace Generation

In [ ]:
# This downloads google/flan-t5-base on first run.
        # answer = engine.generate_answer(rag_question, contexts, backend="hf", model_name="google/flan-t5-base")
        # print(answer)

## 10. Gradio UI

In [ ]:
import gradio as gr

        def ask(query, mode, top_k, use_hf):
            if mode == "BM25":
                results = engine.bm25_search(query, top_k=int(top_k))
            else:
                results = engine.hybrid_search(query, top_k=int(top_k))
            backend = "hf" if use_hf else "extractive"
            try:
                answer = engine.generate_answer(query, results, backend=backend)
            except Exception as exc:
                answer = engine.generate_answer(query, results, backend="extractive") + f"\n\nFallback: {exc}"
            rows = [[r.rank, r.doc.doc_id, r.doc.title, round(r.score, 4), r.doc.link] for r in results]
            return answer, rows

        with gr.Blocks(title="StackLite Hybrid QA") as demo:
            gr.Markdown("# StackLite Hybrid QA")
            query = gr.Textbox(label="Question", lines=2)
            mode = gr.Radio(["Hybrid RRF", "BM25"], value="Hybrid RRF", label="Retrieval")
            top_k = gr.Slider(3, 10, value=5, step=1, label="Citations")
            use_hf = gr.Checkbox(value=False, label="Use HuggingFace FLAN-T5")
            button = gr.Button("Ask")
            answer = gr.Textbox(label="Answer with citations", lines=8)
            sources = gr.Dataframe(headers=["rank", "doc_id", "title", "score", "url"], label="Retrieved sources")
            button.click(ask, [query, mode, top_k, use_hf], [answer, sources])

        demo.launch(share=True)